In [3]:
import gdsfactory as gf


from mesalab_pdk import get_pdk, LAYER

pdk = get_pdk()
pdk.activate()

In [4]:
from directional_couplers import coupler_imgrev
from directional_couplers import dc_lut
from cross_sections import xs_ekn300_te_IMGREV, xs_heater_metal

#dc_lut = DirectionalCouplerLUT("static/directional_coupler_lut.csv")
Ldc = dc_lut.get_length_um(gap_nm = 300, width_um=0.75, ratio=0.5)
local_xs = gf.get_cross_section(xs_ekn300_te_IMGREV, width = 0.5)

dc = gf.partial(coupler_imgrev, gap = 0.3, length=Ldc, dy = 127, dx = 500, cross_section=local_xs, layer_core="WG", layer_trench="SIN_ETCH")
ekn_bend=gf.partial(gf.c.bend_euler, cross_section=local_xs)

# mm = gf.components.mzi(splitter=dc, combiner=dc, cross_section=xs_ekn300_te_IMGREV,
#                        port_e0_splitter = "o2", port_e1_splitter = "o1",
#                        port_e0_combiner = "o2", port_e1_combiner = "o1",
#                        length_y = 10, length_x = 500, bend = ekn_bend).show()


mm = gf.components.mzi2x2_2x2_phase_shifter(splitter=dc, combiner=dc, cross_section=local_xs,
                       port_e0_splitter = "o2", port_e1_splitter = "o1",
                       port_e0_combiner = "o2", port_e1_combiner = "o1",
                       length_y = 10, length_x = 500, bend = ekn_bend).show()

In [ ]:
from directional_couplers import coupler_ring_imgrev, coupler_imgrev
from cross_sections import xs_ekn300_te_IMGREV

xs = xs_ekn300_te_IMGREV(
    width=0.75,
    width_trench=50,
    layer="WG",
    layer_trench="SIN_ETCH",
    radius_min=10, radius=10,
)
d = gf.Component()

# c = coupler_ring_imgrev(
#     gap=0.1,
#     radius=300,
#     length_x=25,
#     cross_section=xs,
#     layer_trench='SIN_ETCH',
#     layer_core='WG'
# )

# cref = d.add_ref(c).dmovex(-1300)

#f = gf.components.coupler(dx=500, dy=127, cross_section=xs_ekn300_te_IMGREV, gap=0.5, length= 50)
e = coupler_imgrev(dx = 500, dy = 127, cross_section=xs_ekn300_te_IMGREV, gap=0.5, length= 50, centered=True)

d.add_ref(e)
#d.add_ref(f) 
d.show()

In [ ]:
from __future__ import annotations

from collections.abc import Iterable
import gdsfactory as gf
from gdsfactory.component import Component
from gdsfactory.typings import ComponentSpec, CrossSectionSpec, LayerSpec


def _copy_ports(dst: Component, src: Component) -> None:
    for port in src.ports:
        dst.add_port(name=port.name, port=port)


def _copy_metadata(dst: Component, src: Component) -> None:
    #dst.info.update(src.info)
    try:
        dst.copy_child_info(src)
    except Exception:
        pass


def _get_section_layer(
    xs: gf.CrossSection,
    section_names: Iterable[str] = ("trench_top", "trench_bot"),
) -> LayerSpec:
    names = set(section_names)
    for section in xs.sections:
        if section.name in names:
            return section.layer

    raise ValueError(
        f"Could not infer trench layer. Expected one of {tuple(names)} "
        "in cross_section.sections."
    )


def resolve_imgrev_layers(
    cross_section: CrossSectionSpec,
    layer_core: LayerSpec | None = None,
    layer_trench: LayerSpec | None = None,
    trench_section_names: tuple[str, ...] = ("trench_top", "trench_bot"),
) -> tuple[LayerSpec, LayerSpec]:
    xs = gf.get_cross_section(cross_section)

    if layer_core is None:
        layer_core = xs.layer

    if layer_trench is None:
        layer_trench = _get_section_layer(xs, trench_section_names)

    return layer_core, layer_trench


def postprocess_imgrev_trenches(
    component: Component,
    cross_section: CrossSectionSpec,
    layer_core: LayerSpec | None = None,
    layer_trench: LayerSpec | None = None,
    trench_section_names: tuple[str, ...] = ("trench_top", "trench_bot"),
    keep_other_layers: bool = True,
) -> Component:
    """Converts normal trench-defined waveguide geometry into image-reversal-safe geometry.

    Operation:
        1. flatten a copy of the component
        2. extract core layer
        3. extract trench layer
        4. subtract core from trench
        5. keep ports and metadata at top level
    """
    layer_core, layer_trench = resolve_imgrev_layers(
        cross_section=cross_section,
        layer_core=layer_core,
        layer_trench=layer_trench,
        trench_section_names=trench_section_names,
    )

    src = gf.get_component(component)

    flat = Component()
    flat_ref = flat << src
    flat.flatten()

    # core = flat.extract(layers=[layer_core])
    # trench = flat.extract(layers=[layer_trench])

    

    trench_clean = gf.boolean(
        A=flat,
        B=flat,
        operation="not",
        layer=layer_trench,
        layer1=layer_trench,
        layer2=layer_core
    )

    #trench_clean.show()

    out = Component()

    if keep_other_layers:
        other_layers = [
            layer
            for layer in flat.layers
            if gf.get_layer(layer) not in {
                gf.get_layer(layer_trench),
            }
        ]

        if other_layers:
            out << flat.extract(layers=other_layers)

    # out << core
    out << trench_clean

    _copy_ports(out, src)
    _copy_metadata(out, src)

    return out

@gf.cell
def coupler_imgrev(
    gap: float = 0.2,
    length: float = 20.0,
    dy: float = 4.0,
    dx: float = 10.0,
    cross_section: CrossSectionSpec = "xs_ekn300_te_IMGREV",
    layer_core: LayerSpec | None = None,
    layer_trench: LayerSpec | None = None,
) -> Component:
    base = gf.components.coupler(
        gap=gap,
        length=length,
        dy=dy,
        dx=dx,
        cross_section=cross_section,
    )

    return postprocess_imgrev_trenches(
        component=base,
        cross_section=cross_section,
        layer_core=layer_core,
        layer_trench=layer_trench,
    )


@gf.cell
def coupler_ring_imgrev(
    gap: float = 0.2,
    radius: float | None = None,
    length_x: float = 4.0,
    bend: ComponentSpec = "bend_euler",
    straight: ComponentSpec = "straight",
    cross_section: CrossSectionSpec = "xs_ekn300_te_IMGREV",
    cross_section_bend: CrossSectionSpec | None = None,
    length_extension: float | None = None,
    layer_core: LayerSpec | None = None,
    layer_trench: LayerSpec | None = None,
) -> Component:
    base = gf.components.coupler_ring(
        gap=gap,
        radius=radius,
        length_x=length_x,
        bend=bend,
        straight=straight,
        cross_section=cross_section,
        cross_section_bend=cross_section_bend,
        length_extension=length_extension,
    )

    return postprocess_imgrev_trenches(
        component=base,
        cross_section=cross_section,
        layer_core=layer_core,
        layer_trench=layer_trench,
    )



In [ ]:
from arrays import array_with_y_span

c = array_with_y_span(
    components=(
        gf.components.straight(length=100),
        gf.components.straight(length=150),
        gf.components.straight(length=200),
    ),
    pitch_x=400,
    y_span=300,
    labels=("L100\n555", "L150", "L200"),
    label_offset=(200, 0),
    label_rotation=0,
)

c.show()